In [1]:
# Add project root to Python path and change working directory
import sys
import os
from pathlib import Path

# Get project root
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Change working directory to project root so config.yaml can be found
os.chdir(project_root)

print(f"Added to path: {project_root}")
print(f"Working directory: {os.getcwd()}")

Added to path: c:\Users\J'sen\Desktop\GitHub\finance-rag
Working directory: c:\Users\J'sen\Desktop\GitHub\finance-rag


### Docling API Test
---



In [ ]:
from src.preprocessing.document_parser import DocumentParser
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.types.doc import PictureItem, TableItem

In [ ]:
# Configure Docling pipeline options
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA
)

pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_table_structure = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF, InputFormat.DOCX, InputFormat.HTML],
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)
file_path = "./src/data/pdf/AAPL/10-K/AAPL_10-K_0000320193-25-000079.pdf"
# file_path = "./src/data/pdf/sample/sample-unstructured-paper.pdf"
result = converter.convert(file_path)
doc = result.document

In [ ]:
from io import BytesIO
import base64
from PIL import Image

def pil_to_base64(image) -> str:
    """
        Convert a PIL Image to a base64 string.
        Args:
            image: PIL Image object
        Returns:
            Base64-encoded image string (utf-8)
    """

    buffer = BytesIO()
    image.save(buffer, format="PNG")

    img_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

    return img_base64


In [ ]:
from IPython.display import display, Markdown
from src.utils.models import load_vision_model

vision_client = load_vision_model()
tables_processed = 0
max_tables = 5

for idx, (item,level) in enumerate(doc.iterate_items()):
    # Extract page number from provenance
    page_number = None
    if hasattr(item, 'prov') and item.prov:
        page_refs = [p.page_no for p in item.prov if hasattr(p, 'page_no')]
        page_number = page_refs[0] if page_refs else None

    # Extract table image if item is a table
    if isinstance(item, TableItem):
        table_image = item.get_image(doc)
    
        display(table_image)
        
        table_md = item.export_to_markdown(doc)
        display(Markdown(table_md))

        prompt = (
            "Below is a markdown table extracted via OCR from a "
            "financial document. Compare it against the table image "
            "and correct any OCR errors, missing values, or "
            "formatting issues. Return ONLY the corrected markdown "
            f"table, nothing else.\n\n{table_md}"
        )
        response = vision_client.describe_image(table_image, prompt)
        display(Markdown(response))

        tables_processed += 1
        if tables_processed >= max_tables:
            break  # Stop after processing the desired number of tables
  

### Batch Processing
---


In [2]:
from src.utils.logger import get_logger
from src.preprocessing.chunking import SemanticChunker
from src.indexing.vector_store import get_vector_store
from src.indexing.embedder import get_embedder
from src.utils.utils import extract_metadata_from_filename

logger = get_logger(__name__)

def get_pdf_files(folder_path: str) -> list[str]:
    """Get all PDF files from a folder as strings."""
    folder = Path(folder_path)
    return [str(pdf) for pdf in folder.glob("*.pdf")]


def ingest_batch(file_paths: list):
    """
    Handle batch document ingestion from list of file paths.
    Automatically extracts metadata from filenames.
    
    Args:
        file_paths: List of file paths to ingest
    """
    logger.info(f"Starting batch ingestion of {len(file_paths)} documents")
    
    chunker = SemanticChunker()
    
    # Extract metadata from file paths
    file_metadata = []
    for file_path in file_paths:
        try:
            metadata = extract_metadata_from_filename(file_path)
            file_metadata.append(metadata)
            logger.info(f"Extracted: {metadata['company']} {metadata['document_type']} from {Path(file_path).name}")
        except ValueError as e:
            logger.warning(f"Skipping {file_path}: {e}")
    
    if not file_metadata:
        logger.error("No valid files to process")
        return
    
    # Batch chunk all documents
    doc_chunks = chunker.chunk_documents_batch(file_metadata)
    
    logger.info(f"Generated {len(doc_chunks)} chunks from {len(file_metadata)} documents")
    
    store = get_vector_store()
    embedder = get_embedder()
    
    # Batch embed and upsert
    embeddings = embedder.embed_batch([chunk.content for chunk in doc_chunks])
    store.upsert(doc_chunks, embeddings)
    
    logger.info(f"Collection has {store.count()} points")


c:\Users\J'sen\Desktop\GitHub\finance-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
file_paths = get_pdf_files("./src/data/filings_to_process")
print(file_paths)

ingest_batch(file_paths)